In [1]:
from sqlalchemy import create_engine, text

engine = create_engine(
    "mssql+pyodbc://localhost/Alex_weather?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server&TrustServerCertificate=yes"
)

In [2]:
steps = [
    "truncate table gold.Fact_Weather_Hourly",

    """
    insert into gold.Fact_Weather_Hourly
    select
      d.Date_SK,
      datepart(hour, s.time),
      l.Location_SK,
      s.temperature_2m,
      s.precipitation,
      s.wind_gusts_10m,
      s.pressure_msl
    from silver.weather s
    join gold.dim_location1 l on l.Location_Name = s.location
    join gold.Dim_Date d      on d.Full_Date = cast(s.time as date)
    """,

    "drop table if exists gold.Fact_Risk",

    """
    select
      Date_SK,
      Location_SK,
      max(Wind_Gusts_10m) as Max_Gust,
      case when max(Wind_Gusts_10m) >= 65 then 'High' else 'Low' end as Risk_Level
    into gold.Fact_Risk
    from gold.Fact_Weather_Hourly
    group by Date_SK, Location_SK
    """,
]

with engine.begin() as conn:
    for sql in steps:
        conn.execute(text(sql))

print("Gold done")

Gold done


In [4]:
with engine.connect() as conn:
    print("Fact_Weather_Hourly:", conn.execute(text("select count(*) from gold.Fact_Weather_Hourly")).scalar())
    print("Fact_Risk:", conn.execute(text("select count(*) from gold.Fact_Risk")).scalar())

Fact_Weather_Hourly: 6480
Fact_Risk: 270
